# SafeSight AI — PPE Model Training

**Run this notebook on Google Colab with a GPU runtime.**

Go to: `Runtime → Change runtime type → T4 GPU` before running.

---

**What this does:**
1. Checks the GPU
2. Installs dependencies
3. Downloads the PPE dataset from Roboflow
4. Fine-tunes YOLO26s for 80 epochs (~2 hrs on T4)
5. Downloads the trained weights to your computer

**You need:** A free Roboflow API key from [roboflow.com](https://roboflow.com)

In [ ]:
# ── Step 1: Verify GPU ────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → T4 GPU and re-run."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu}  ({vram:.1f} GB VRAM)")
print("Ready to train.")

In [ ]:
# ── Step 2: Install dependencies ──────────────────────────────────────────────
%pip install -q ultralytics roboflow
print("Done.")

In [ ]:
# ── Step 3: Download PPE dataset from Roboflow ────────────────────────────────
# Paste your free API key from https://roboflow.com (Account → Roboflow API Key)
ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"   # <-- edit this

if ROBOFLOW_API_KEY == "PASTE_YOUR_KEY_HERE":
    raise ValueError("Please paste your Roboflow API key above before running.")

from roboflow import Roboflow
from pathlib import Path

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
dataset = project.version(1).download("yolov8", location="data/ppe_dataset")

DATA_YAML = Path("data/ppe_dataset/data.yaml")
print(f"\nDataset ready: {DATA_YAML}")
print(f"Classes: {dataset.classes}")

In [ ]:
# ── Step 4: Preview dataset ───────────────────────────────────────────────────
import yaml
cfg = yaml.safe_load(DATA_YAML.read_text())
print("Classes:", cfg.get('names'))

# Count images
for split in ['train', 'valid', 'test']:
    p = Path(f"data/ppe_dataset/{split}/images")
    if p.exists():
        n = len(list(p.glob('*')))
        print(f"  {split}: {n} images")

In [ ]:
# ── Step 5: Train ─────────────────────────────────────────────────────────────
# ~2 hours on T4 GPU. Checkpoints save every 10 epochs.
# If Colab disconnects, re-run with resume=True (cell below).

from ultralytics import YOLO

model = YOLO("yolo26s.pt")  # auto-downloads base weights

results = model.train(
    data        = str(DATA_YAML),
    epochs      = 80,
    imgsz       = 1280,
    batch       = 16,       # T4 has 16 GB — bumped from local 8
    workers     = 4,
    device      = 0,
    project     = "models",
    name        = "safesight-ppe",
    save        = True,
    save_period = 10,
    plots       = True,
    # Augmentation — same as config/train.yaml
    degrees     = 15.0,
    perspective = 0.0008,
    flipud      = 0.05,
    fliplr      = 0.5,
    scale       = 0.6,
    translate   = 0.15,
    shear       = 4.0,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.05,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    erasing     = 0.3,
    lr0         = 0.001,
    lrf         = 0.01,
    warmup_epochs = 3,
    weight_decay  = 0.0005,
    close_mosaic  = 10,
    val         = True,
)

print("\nTraining complete!")

In [ ]:
# ── Step 5b: Resume if Colab disconnected ─────────────────────────────────────
# Only run this cell if training was interrupted. Skip otherwise.

from ultralytics import YOLO
from pathlib import Path

last = Path("models/safesight-ppe/weights/last.pt")
if not last.exists():
    raise FileNotFoundError(f"No checkpoint at {last} — run Step 5 from scratch.")

print(f"Resuming from {last}")
model = YOLO(str(last))
results = model.train(resume=True)

In [ ]:
# ── Step 6: Evaluate best model ───────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

best = Path("models/safesight-ppe/weights/best.pt")
model = YOLO(str(best))
metrics = model.val(data=str(DATA_YAML))

print(f"\nmAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")
print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")

In [ ]:
# ── Step 7: Show training curves ──────────────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

for img in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    p = Path(f"models/safesight-ppe/{img}")
    if p.exists():
        print(f"\n{img}")
        display(Image(str(p)))

In [ ]:
# ── Step 8: Download weights to your computer ─────────────────────────────────
# This will pop a download dialog in your browser.
# Save the file as: safesight-ai/models/safesight-ppe.pt

import shutil
from pathlib import Path
from google.colab import files

best = Path("models/safesight-ppe/weights/best.pt")
out  = Path("safesight-ppe.pt")
shutil.copy(best, out)

print(f"File size: {out.stat().st_size / 1e6:.1f} MB")
files.download(str(out))
print("\nDownload started. Save to:  safesight-ai/models/safesight-ppe.pt")
print("Then run the pipeline with: python -m src.pipeline --source YOUR_SOURCE --model models/safesight-ppe.pt")